# Climate Explorer: Comparing US Cities

In this notebook, we'll compare **5 US cities** with very different climates. Just as birdsong researchers learn to identify species from spectrogram shapes, you'll learn to identify **climate types** from temperature patterns.

| City | Station ID | Climate Type |
|------|-----------|-------------|
| Miami, FL | USW00012839 | Tropical |
| New York, NY | USW00094728 | Humid Continental |
| Chicago, IL | USW00094846 | Continental |
| Los Angeles, CA | USW00023174 | Mediterranean |
| Seattle, WA | USW00024233 | Marine |

## Step 1: Import Libraries

In [ ]:
import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

try:
    import folium
except ImportError:
    !pip install folium
    import folium

## Step 2: Fetch All 5 Stations

In [ ]:
def fetch_station(station_id, name):
    """Fetch daily temperature data from NOAA for a given station."""
    url = "https://www.ncei.noaa.gov/access/services/data/v1"
    params = {
        "dataset": "daily-summaries",
        "stations": station_id,
        "startDate": "2000-01-01",
        "endDate": "2024-12-31",
        "dataTypes": "TMAX,TMIN,TAVG",
        "units": "metric",
        "format": "json"
    }
    r = requests.get(url, params=params, timeout=60)
    r.raise_for_status()
    df = pd.DataFrame(r.json())

    df["DATE"] = pd.to_datetime(df["DATE"])
    for col in ["TAVG", "TMAX", "TMIN"]:
        df[col] = pd.to_numeric(df[col], errors="coerce")
    df["TAVG"] = df["TAVG"].fillna((df["TMAX"] + df["TMIN"]) / 2)
    df["year"] = df["DATE"].dt.year
    df["month"] = df["DATE"].dt.month
    df["dayofyear"] = df["DATE"].dt.dayofyear

    print(f"  {name}: {len(df)} records ({df['DATE'].min().date()} to {df['DATE'].max().date()})")
    return df

In [ ]:
import time

stations = {
    "Miami":       "USW00012839",
    "New York":    "USW00094728",
    "Chicago":     "USW00094846",
    "Los Angeles": "USW00023174",
    "Seattle":     "USW00024233",
}

print("Fetching data from NOAA...")
station_data = {}
for name, sid in stations.items():
    try:
        station_data[name] = fetch_station(sid, name)
    except Exception as e:
        print(f"  {name}: FAILED — {e}")
    time.sleep(1)
print(f"Done! Fetched {len(station_data)}/{len(stations)} stations.")

## Step 3: Monthly Climatology — Seasonal Profiles

Each city has a characteristic seasonal shape. Compare how the curves differ.

In [ ]:
month_names = ["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"]
colors = {"Miami": "#e74c3c", "New York": "#3498db", "Chicago": "#2ecc71",
          "Los Angeles": "#f39c12", "Seattle": "#9b59b6"}

plt.figure(figsize=(10, 6))
for name, df in station_data.items():
    monthly = df.groupby("month")["TAVG"].mean()
    plt.plot(range(1, 13), monthly.values, "o-", label=name, color=colors[name], linewidth=2)

plt.xticks(range(1, 13), month_names)
plt.title("Monthly Climatology: 5 US Cities (2000–2024)")
plt.xlabel("Month")
plt.ylabel("Temperature (°C)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print("Notice: Miami is warm year-round. Chicago has the widest swing.")
print("Los Angeles and Seattle are moderate but have different seasonal shapes.")

## Step 4: Heatmap Montage

Each heatmap shows one city's full 25-year temperature record. Compare the patterns:
- **Uniformly warm** = tropical
- **Wide color range** = continental (hot summers, cold winters)
- **Narrow color range** = marine/Mediterranean

In [ ]:
fig, axes = plt.subplots(5, 1, figsize=(15, 22))

vmin, vmax = -25, 35  # Same scale for all cities (includes cold continental winters)

for ax, (name, df) in zip(axes, station_data.items()):
    pivot = df.pivot_table(index="year", columns="dayofyear", values="TAVG")
    sns.heatmap(pivot, cmap="coolwarm", vmin=vmin, vmax=vmax, ax=ax,
                cbar_kws={"label": "°C"}, xticklabels=30, yticklabels=2)
    ax.set_title(name, fontsize=14, fontweight="bold")
    ax.set_xlabel("Day of Year")
    ax.set_ylabel("Year")

plt.tight_layout()
plt.show()

## Step 5: Annual Average Comparison

Overlay all 5 cities on one chart to see long-term trends.

In [ ]:
plt.figure(figsize=(12, 6))

for name, df in station_data.items():
    annual = df.groupby("year")["TAVG"].mean()
    plt.plot(annual.index, annual.values, "o-", label=name, color=colors[name], markersize=4)

plt.title("Annual Average Temperature: 5 US Cities (2000–2024)")
plt.xlabel("Year")
plt.ylabel("Temperature (°C)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## Step 6: Daily Temperature Range

The **daily range** (TMAX - TMIN) reveals how much temperature swings within a single day. Continental climates have larger ranges; maritime climates are more stable.

In [ ]:
range_data = {}
for name, df in station_data.items():
    daily_range = (df["TMAX"] - df["TMIN"]).dropna()
    range_data[name] = daily_range.mean()

cities = list(range_data.keys())
ranges = list(range_data.values())

plt.figure(figsize=(8, 5))
plt.bar(cities, ranges, color=[colors[c] for c in cities])
plt.title("Mean Daily Temperature Range (TMAX - TMIN)")
plt.ylabel("Temperature Range (°C)")
plt.grid(axis="y", alpha=0.3)
plt.show()

for c, r in range_data.items():
    print(f"  {c}: {r:.1f} °C daily range")

## Step 7: Mystery Station Challenge

Below are heatmaps from two **unlabeled** stations. Based on what you've learned, can you identify their climate type?

Look for clues:
- How wide is the seasonal swing?
- How warm are the summers? How cold are the winters?
- Is the pattern uniform or highly variable?

In [ ]:
# Mystery stations — can you guess which cities these are?
mystery = {
    "Mystery A": station_data[list(station_data.keys())[2]],
    "Mystery B": station_data[list(station_data.keys())[0]],
}

fig, axes = plt.subplots(2, 1, figsize=(15, 9))
for ax, (label, df) in zip(axes, mystery.items()):
    pivot = df.pivot_table(index="year", columns="dayofyear", values="TAVG")
    sns.heatmap(pivot, cmap="coolwarm", vmin=-25, vmax=35, ax=ax,
                cbar_kws={"label": "°C"}, xticklabels=30, yticklabels=2)
    ax.set_title(label, fontsize=14, fontweight="bold")
    ax.set_xlabel("Day of Year")
    ax.set_ylabel("Year")

plt.tight_layout()
plt.show()

print("Can you guess which cities these are? Scroll down for the answer...")

In [ ]:
# === ANSWERS ===
mystery_a_name = list(station_data.keys())[2]
mystery_b_name = list(station_data.keys())[0]

print(f"Mystery A = {mystery_a_name}")
print("  Clue: Extreme seasonal swing, cold winters (deep blue), hot summers")
print()
print(f"Mystery B = {mystery_b_name}")
print("  Clue: Uniformly warm year-round, narrow color range, tropical pattern")

## Step 8: Station Map

In [ ]:
station_coords = {
    "Miami":       (25.793, -80.290),
    "New York":    (40.782, -73.965),
    "Chicago":     (41.881, -87.623),
    "Los Angeles": (33.938, -118.389),
    "Seattle":     (47.449, -122.309),
}

m = folium.Map(location=[38, -96], zoom_start=4)

for name, (lat, lon) in station_coords.items():
    mean_t = station_data[name]["TAVG"].mean()
    folium.Marker(
        location=[lat, lon],
        popup=f"{name}: {mean_t:.1f} °C",
        tooltip=name,
        icon=folium.Icon(color="red", icon="info-sign")
    ).add_to(m)

m

## What's Next

In **Notebook 3**, we'll teach a **neural network** to automatically distinguish climate types using machine learning — the same autoencoder technique used in birdsong research.